In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

Inputs

In [2]:
features = pd.read_parquet(variables.FEATURES_UNIQUE_DATA_PATH)
targets = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)
features_ranked_ordered_unique = pd.read_parquet(variables.FEATURES_RANKED_ORDERED_UNIQUE_PATH)

In [3]:
print(features.shape)
print(targets.shape)
print(features_ranked_ordered_unique.shape)

(20000, 466)
(736417, 96)
(466, 96)


Subset sample

In [4]:
# Randomly sample index
seed = np.random.default_rng(44)
index = seed.choice(len(features), size=variables.FEATURE_SELECTION_SUBSAMPLE_AMOUNT, replace=False)
index.sort()

features = features.iloc[index]
targets = targets.iloc[index]

In [5]:
# Split feature data into x parts of train / validation parts
multiple_train_validation_subsets_index_ranges = TimeSeriesSplit(n_splits=3).split(features)
multiple_train_validation_subsets_index_ranges = list(multiple_train_validation_subsets_index_ranges)

In [6]:
def generate_feature_size_search_space():
    max_num_of_features = features.shape[1]
    num_of_features_to_test_per_horizon = np.linspace(start=10, stop=max_num_of_features, num=10).round(0)
    display(num_of_features_to_test_per_horizon)
    return num_of_features_to_test_per_horizon

num_of_features_to_test_per_horizon = generate_feature_size_search_space()

array([ 10.,  61., 111., 162., 213., 263., 314., 365., 415., 466.])

In [7]:
train_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "min_child_samples": 20,
    "bagging_fraction": 0.5,
    "bagging_freq": 1,
    "force_col_wise": True,
    "random_state": 42,
    "n_jobs": 1,
    "num_threads": 1,
    "verbose": -1,
}

Core logic 

Train a LGB model per horizon, per top number of features, per subset fold. Extract MAE of each model, which determines best number of top features for each horizon

models = (CV folds=3)  *  (horizons=96) * (top feature n search space = 10)

models = 2880

In [8]:
features_ranked_ordered_unique[:10]

,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,...,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
feature,,,,,,,,,,,,,,,,,,,,,
nsw_price_asinh_lag_1,6,4,4,4,4,2,2,4,4,4,...,92,95,95,110,102,108,89,115,112,91
nsw_price_asinh_lag_2,9,6,6,6,6,6,6,6,6,6,...,79,94,109,89,77,93,109,111,98,104
nsw_price_asinh_lag_4,13,12,12,12,12,12,12,11,9,11,...,85,104,118,102,120,113,119,110,118,111
nsw_price_asinh_lag_12,26,23,22,21,19,23,22,24,25,28,...,117,117,121,112,100,87,112,113,116,115
nsw_price_asinh_lag_48,159,154,152,148,144,150,150,146,134,141,...,93,91,99,78,85,80,82,84,76,80
nsw_price_asinh_lag_96,235,223,224,214,212,212,211,208,197,208,...,118,123,107,123,123,135,125,136,117,122
nsw_price_asinh_lag_336,211,198,198,191,195,192,199,180,194,205,...,126,137,125,129,130,125,134,126,134,120
nsw_price_asinh_lag_335,202,199,186,195,193,196,202,204,195,188,...,133,131,134,131,138,134,135,117,139,128
nsw_price_asinh_lag_337,205,196,184,207,194,202,189,193,188,195,...,130,120,126,118,125,120,127,131,137,134


In [9]:
def find_num_best_features_for_horizon(horizon_index):
    # Select the feature ranking scores for this horizon only
    top_feature_for_this_horizon = features_ranked_ordered_unique.iloc[:, horizon_index]
    # Explicitly sort by score high to low and use feature names from the index
    top_feature_for_this_horizon = top_feature_for_this_horizon.sort_values(ascending=False)
    
    # Filter the underlying features dataset by the ordered top features
    all_features_for_this_horizon = features.loc[:, top_feature_for_this_horizon.index]

    MAE_per_fold_per_top_feature_amount = []

    # For each split (which as a train and valid component)
    for train_index, valid_index in multiple_train_validation_subsets_index_ranges:
        subset_x_features_train = all_features_for_this_horizon.iloc[train_index]
        subset_x_features_valid = all_features_for_this_horizon.iloc[valid_index]
        subset_x_targets_train = targets.iloc[train_index, horizon_index]
        subset_x_targets_valid = targets.iloc[valid_index, horizon_index]

        # For each number of max features to test 
        for top_n_features in num_of_features_to_test_per_horizon.astype(int):
            
            
            features_for_this_horizon_top = subset_x_features_train.iloc[:, :top_n_features]
            valid_features_for_this_horizon_top = subset_x_features_valid.iloc[:, :top_n_features]
            train_data = lgb.Dataset(features_for_this_horizon_top, label=subset_x_targets_train, free_raw_data=False)
            model = lgb.train(train_params, train_data, num_boost_round=200)

            y_pred = model.predict(valid_features_for_this_horizon_top)
            average_prediction_error = np.mean(np.abs(subset_x_targets_valid - y_pred))

            MAE_per_fold_per_top_feature_amount.append((top_n_features, average_prediction_error))

            del train_data, model, y_pred

        del subset_x_features_train, subset_x_features_valid, subset_x_targets_train, subset_x_targets_valid

    return horizon_index, MAE_per_fold_per_top_feature_amount

results = run_parallel(
    function=find_num_best_features_for_horizon,
    data=list(range(variables.HORIZON_COUNT)),
)

Initialising workers..:   0%|          | 0/5 [00:00<?, ?step/s]

os=linux cpu_count=48 available_memory_gb=177.24 memory_per_worker_gb=3.74 max_assigned_workers=45


Initialising workers..: 100%|██████████| 5/5 [12:15<00:00, 147.13s/step]


Clean up

In [10]:
model_results = pd.DataFrame(results, columns=["horizon", "maes"]).explode("maes").reset_index(drop=True)
model_results[["n_features", "mae"]] = pd.DataFrame(model_results.pop("maes").tolist(), index=model_results.index)
model_results_best = model_results.loc[model_results.groupby("horizon")["mae"].idxmin()].reset_index(drop=True)

In [11]:
# Create a dataframe mapping table for included features for each horizon

horizon_columns = [f"h{i + 1}" for i in range(variables.HORIZON_COUNT)]
feature_horizon_matrix = pd.DataFrame(False, index=features_ranked_ordered_unique.index, columns=horizon_columns)

for _, row in model_results_best.iterrows():
    horizon_index = int(row["horizon"])
    n_features = int(row["n_features"])
    selected = features_ranked_ordered_unique.iloc[:, horizon_index].sort_values(ascending=False).index[:n_features]
    feature_horizon_matrix.loc[selected, f"h{horizon_index + 1}"] = True

feature_horizon_matrix = feature_horizon_matrix.reset_index().rename(columns={"index": "feature"})


View

In [18]:
pd.set_option("display.max_rows", None)
model_results_best[:100]

,horizon,n_features,mae
0,0,10,27.947880
1,1,10,32.519592
2,2,10,32.198879
3,3,10,33.816323
4,4,10,36.379428
5,5,10,35.364521
6,6,10,28.788665
7,7,415,29.993777
8,8,466,27.701687
9,9,263,25.471950


In [13]:
display(feature_horizon_matrix[:20])

,feature,h1,h2,h3,h4,h5,h6,h7,h8,h9,...,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_asinh_lag_1,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,nsw_price_asinh_lag_2,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,nsw_price_asinh_lag_4,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,nsw_price_asinh_lag_12,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,nsw_price_asinh_lag_48,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
5,nsw_price_asinh_lag_96,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
6,nsw_price_asinh_lag_336,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
7,nsw_price_asinh_lag_335,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
8,nsw_price_asinh_lag_337,False,False,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,False
9,nsw_price_asinh_rmean_48,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


Export

In [14]:
model_results_best.to_csv("model_results_best.csv")

In [15]:
feature_horizon_matrix.to_parquet(variables.FEATURES_OPTIMAL_AMOUNT_PATH)